In [ ]:
# CPU allocation and project bootstrap
import os
import sys
from pathlib import Path
import multiprocessing as mp

CPU_START = 0
CPU_STOP = 10
CPU_RANGE = range(CPU_START, min(CPU_STOP, mp.cpu_count()))
CPU_USE = max(1, len(list(CPU_RANGE)))

os.environ["MY_CPU_COUNT"] = str(CPU_USE)
os.environ["OMP_NUM_THREADS"] = str(CPU_USE)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_USE)
os.environ["MKL_NUM_THREADS"] = str(CPU_USE)
os.environ["NUMEXPR_MAX_THREADS"] = str(CPU_USE)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_markov_channel_negativity")

try:
    os.sched_setaffinity(0, set(CPU_RANGE))
except Exception as exc:
    print(f"CPU affinity not set: {exc}")
try:
    print("CPU affinity:", os.sched_getaffinity(0))
except Exception:
    pass

def find_repo_root(start):
    start = Path(start).resolve()
    candidates = [start, *start.parents]
    fallback = Path("/home/abhuiyan/class_A_fermionic_adaptive_circuit")
    if fallback not in candidates:
        candidates.append(fallback)
    for candidate in candidates:
        if (candidate / "REPO_POLICY.md").exists() and (candidate / "src" / "fgtn").exists():
            return candidate
    raise RuntimeError(f"Could not locate repo root from {start}")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(ROOT)
print("ROOT =", ROOT)

In [ ]:
# Imports and run controls
import importlib
import json
import time
from datetime import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import pandas as pd
from IPython.display import HTML, Video, display
from tqdm.auto import tqdm

classA_module = importlib.import_module("fgtn.classA_U1FGTN")
importlib.reload(classA_module)
from fgtn.classA_U1FGTN import classA_U1FGTN

RUN_SMOKE = False
NX = 4 if RUN_SMOKE else 16
NY = 6 if RUN_SMOKE else 24
CYCLES = 2 if RUN_SMOKE else 24
NSHELL = None
DW = True
DW_TRUNCATION = False
ALPHA_1 = 1
ALPHA_2 = 30
INIT_MODE = "default"
SEQUENCE = "raster_y"
N_A = 0.5
PERFECT_CORRECTION = False
DECOH_VALUES = [True, False]
REMEMBER_INIT = True
SAVE_MARKOV_CACHE = True

OUTPUT_DIR = ROOT / "notebooks" / "markov_channel_negativity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    mpl.rcParams["font.family"] = "CMU Sans Serif"
except Exception:
    pass
mpl.rcParams["figure.dpi"] = 120
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["mathtext.fontset"] = "cm"

RUN_CONFIG = {
    "NX": NX,
    "NY": NY,
    "CYCLES": CYCLES,
    "NSHELL": NSHELL,
    "DW": DW,
    "DW_TRUNCATION": DW_TRUNCATION,
    "ALPHA_1": ALPHA_1,
    "ALPHA_2": ALPHA_2,
    "INIT_MODE": INIT_MODE,
    "SEQUENCE": SEQUENCE,
    "N_A": N_A,
    "PERFECT_CORRECTION": PERFECT_CORRECTION,
    "DECOH_VALUES": DECOH_VALUES,
    "canonical_dynamics_entry_point": "classA_U1FGTN.run_markov_channel",
}
print(json.dumps(RUN_CONFIG, indent=2, default=str))

In [ ]:
# Indexing and covariance helpers
def timestamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, sort_keys=True, default=str)


def save_dataframe_pair(df, stem):
    csv_path = OUTPUT_DIR / f"{stem}.csv"
    parquet_path = OUTPUT_DIR / f"{stem}.parquet"
    df.to_csv(csv_path, index=False)
    parquet_status = "not attempted"
    try:
        df.to_parquet(parquet_path, index=False)
        parquet_status = "saved"
    except Exception as exc:
        parquet_status = f"skipped: {type(exc).__name__}: {exc}"
    return {"csv_path": str(csv_path), "parquet_path": str(parquet_path), "parquet_status": parquet_status}


def mode_index(nx, x, y, mu):
    return int(mu) + 2 * int(x) + 2 * int(nx) * int(y)


def strip_mode_indices(nx, ny, y0, ay):
    ys = [(int(y0) + dy) % int(ny) for dy in range(int(ay))]
    return np.asarray(
        [mode_index(nx, x, y, mu) for y in ys for x in range(int(nx)) for mu in (0, 1)],
        dtype=np.int64,
    )


def complement_indices(total, selected):
    mask = np.ones(int(total), dtype=bool)
    mask[np.asarray(selected, dtype=np.int64)] = False
    return np.nonzero(mask)[0].astype(np.int64)


def hermitize(matrix):
    matrix = np.asarray(matrix, dtype=np.complex128)
    return 0.5 * (matrix + matrix.conj().T)


def chord_length(ay, ny):
    ay = np.asarray(ay, dtype=np.float64)
    return (float(ny) / np.pi) * np.sin(np.pi * ay / float(ny))


def charge_map_from_G(G, nx, ny):
    G = hermitize(G)
    c_diag = 0.5 * (1.0 + np.real(np.diag(G)))
    return c_diag.reshape(2, int(nx), int(ny), order="F").sum(axis=0)

In [ ]:
# Fermionic partial-time-reversal logarithmic negativity helpers
PTR_EPS = 1e-12


def majorana_cov_from_u1_commutator(G):
    G = hermitize(G)
    n = G.shape[0]
    gamma = np.zeros((2 * n, 2 * n), dtype=np.complex128)
    re = G.real
    im = G.imag
    gamma[0::2, 0::2] = -im
    gamma[0::2, 1::2] = re
    gamma[1::2, 0::2] = -re
    gamma[1::2, 1::2] = -im
    return 0.5 * (gamma - gamma.T)


def _real_from_complex(value, *, tol=1e-8):
    value = np.asarray(value)
    imag_max = float(np.max(np.abs(value.imag))) if value.size else 0.0
    real_value = np.real(value)
    if imag_max > tol:
        print(f"[warn] dropping imaginary part with max |Im|={imag_max:.3e}")
    return real_value


def fermionic_ptr_log_negativity_from_G(G_full, part_a_modes, *, eps=PTR_EPS):
    G_full = hermitize(G_full)
    total_modes = G_full.shape[0]
    part_a_modes = np.asarray(part_a_modes, dtype=np.int64)
    part_b_modes = complement_indices(total_modes, part_a_modes)
    order = np.concatenate([part_a_modes, part_b_modes])
    G = G_full[np.ix_(order, order)]

    gamma = majorana_cov_from_u1_commutator(G)
    n_a_maj = 2 * len(part_a_modes)
    g11 = gamma[:n_a_maj, :n_a_maj]
    g12 = gamma[:n_a_maj, n_a_maj:]
    g21 = gamma[n_a_maj:, :n_a_maj]
    g22 = gamma[n_a_maj:, n_a_maj:]

    gamma_plus = np.block([[g11, 1j * g12], [1j * g21, -g22]])
    gamma_minus = np.block([[g11, -1j * g12], [-1j * g21, -g22]])
    eye_m = np.eye(gamma.shape[0], dtype=np.complex128)
    solve_mat = eye_m + gamma_plus @ gamma_minus
    rhs = gamma_plus + gamma_minus
    try:
        x = np.linalg.solve(solve_mat, rhs)
    except np.linalg.LinAlgError:
        x = np.linalg.pinv(solve_mat, rcond=eps) @ rhs
    gamma_tilde = 0.5 * (eye_m - x)

    mu = np.linalg.eigvals(gamma_tilde)
    term_ptr = np.sum(np.log(np.sqrt(mu + 0j) + np.sqrt(1.0 - mu + 0j)))

    c = 0.5 * (np.eye(total_modes, dtype=np.complex128) + G)
    lam = np.linalg.eigvalsh(hermitize(c)).real
    lam = np.clip(lam, eps, 1.0 - eps)
    term_norm = 0.5 * np.sum(np.log(lam**2 + (1.0 - lam)**2))
    value_complex = term_ptr + term_norm
    value = float(np.maximum(_real_from_complex(value_complex), 0.0))
    return {
        "log_negativity": value,
        "ptr_imag_abs": float(abs(np.imag(value_complex))),
        "lambda_min": float(np.min(lam)),
        "lambda_max": float(np.max(lam)),
        "pinv_used": bool(not np.all(np.isfinite(x))),
    }


def pure_bipartition_log_negativity_from_G_sub(G_sub, *, eps=PTR_EPS):
    zeta = np.linalg.eigvalsh(hermitize(G_sub)).real
    zeta = np.clip(zeta, -1.0 + eps, 1.0 - eps)
    return float(np.sum(np.log1p(np.sqrt(np.maximum(0.0, 1.0 - zeta**2)))))

In [ ]:
# Fitting and video helpers
def fit_log_chord_curve(curve_df, ny):
    sub = curve_df.sort_values("Ay")
    x = np.log(chord_length(sub["Ay"].to_numpy(dtype=np.float64), ny))
    y = sub["log_negativity_mean"].to_numpy(dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 3:
        return {"slope": np.nan, "slope_err": np.nan, "intercept": np.nan, "r2": np.nan, "fit_points": int(x.size)}
    coeffs, cov = np.polyfit(x, y, 1, cov=True)
    slope = float(coeffs[0])
    intercept = float(coeffs[1])
    slope_err = float(np.sqrt(cov[0, 0])) if np.isfinite(cov[0, 0]) else np.nan
    y_hat = slope * x + intercept
    ss_res = float(np.sum((y - y_hat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot == 0.0 else float(1.0 - ss_res / ss_tot)
    return {"slope": slope, "slope_err": slope_err, "intercept": intercept, "r2": r2, "fit_points": int(x.size)}


def make_charge_video(G_hist, *, nx, ny, title, output_path, seconds_per_cycle=3, fps=1, dw_loc=None):
    charge_maps = np.stack([charge_map_from_G(G, nx, ny) for G in G_hist], axis=0)
    frames = np.repeat(np.arange(charge_maps.shape[0]), int(seconds_per_cycle * fps))
    vmax = float(np.nanmax(charge_maps))
    vmin = float(np.nanmin(charge_maps))
    fig, ax = plt.subplots(figsize=(3.375, 2.7), constrained_layout=True)
    im = ax.imshow(charge_maps[0].T, origin="lower", aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    if dw_loc is not None:
        for x_dw in dw_loc:
            ax.axvline(int(x_dw), color="white", lw=0.8, ls="--")
    txt = ax.text(0.03, 0.97, "", transform=ax.transAxes, ha="left", va="top", color="white", fontsize=8, bbox={"fc": "black", "ec": "none", "alpha": 0.65})
    fig.colorbar(im, ax=ax, label="unit-cell charge")

    def update(frame_idx):
        t = int(frames[frame_idx])
        im.set_data(charge_maps[t].T)
        txt.set_text(f"cycle frame {t}/{charge_maps.shape[0] - 1}")
        return im, txt

    ani = animation.FuncAnimation(fig, update, frames=len(frames), blit=False)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if animation.writers.is_available("ffmpeg"):
        ani.save(output_path, writer=animation.FFMpegWriter(fps=int(fps), codec="libx264"))
        plt.close(fig)
        return {"path": str(output_path), "html": None, "status": "mp4 saved"}
    html = ani.to_jshtml(fps=int(fps))
    html_path = output_path.with_suffix(".html")
    html_path.write_text(html, encoding="utf-8")
    plt.close(fig)
    return {"path": str(html_path), "html": HTML(html), "status": "html saved"}

In [ ]:
# Run or load Markov channel histories
histories = {}
run_records = []

for decoh in DECOH_VALUES:
    label = f"decoh_{'on' if decoh else 'off'}"
    print(f"[{timestamp()}] running {label}")
    model = classA_U1FGTN(
        Nx=NX,
        Ny=NY,
        DW=DW,
        nshell=NSHELL,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        dw_truncation=DW_TRUNCATION,
    )
    result = model.run_markov_channel(
        G_history=True,
        progress=True,
        cycles=CYCLES,
        init_mode=INIT_MODE,
        remember_init=REMEMBER_INIT,
        save=SAVE_MARKOV_CACHE,
        save_suffix=f"_markov_channel_negativity_{label}",
        n_a=N_A,
        sequence=SEQUENCE,
        decoh=decoh,
        perfect_correction=PERFECT_CORRECTION,
    )
    G_hist = np.asarray(result["G_hist"], dtype=np.complex128)
    expected_t = CYCLES + (1 if REMEMBER_INIT else 0)
    if G_hist.shape[0] != expected_t:
        raise RuntimeError(f"{label}: expected {expected_t} history frames, got {G_hist.shape[0]}")
    histories[label] = {"G_hist": G_hist, "G_final": np.asarray(result["G_final"], dtype=np.complex128), "model": model, "result": result}
    run_records.append({"label": label, "decoh": bool(decoh), "save_path": result.get("save_path"), "T": int(result["T"]), "cycles": int(result["cycles"])})

run_manifest = {"config": RUN_CONFIG, "runs": run_records, "created": timestamp()}
write_json(OUTPUT_DIR / "run_manifest.json", run_manifest)
pd.DataFrame(run_records)

In [ ]:
# Final-state strip negativity tables
negativity_records = []
curve_records = []
fit_records = []

for label, payload in histories.items():
    print(f"[{timestamp()}] negativity scan {label}")
    G_final = payload["G_final"]
    if G_final.shape != (2 * NX * NY, 2 * NX * NY):
        raise ValueError(f"{label}: unexpected final shape {G_final.shape}")
    for ay in tqdm(range(1, NY // 2 + 1), desc=f"Ay scan {label}", unit="Ay"):
        values = []
        for y0 in range(NY):
            part = strip_mode_indices(NX, NY, y0, ay)
            neg = fermionic_ptr_log_negativity_from_G(G_final, part)
            values.append(float(neg["log_negativity"]))
            negativity_records.append({
                "label": label,
                "decoh": label == "decoh_on",
                "Ay": int(ay),
                "y0": int(y0),
                "log_chord": float(np.log(chord_length(ay, NY))),
                **neg,
            })
        values = np.asarray(values, dtype=np.float64)
        curve_records.append({
            "label": label,
            "decoh": label == "decoh_on",
            "Ay": int(ay),
            "log_chord": float(np.log(chord_length(ay, NY))),
            "log_negativity_mean": float(np.mean(values)),
            "log_negativity_std": float(np.std(values, ddof=1)) if values.size > 1 else 0.0,
            "log_negativity_sem": float(np.std(values, ddof=1) / np.sqrt(values.size)) if values.size > 1 else 0.0,
            "y0_count": int(values.size),
        })

negativity_df = pd.DataFrame(negativity_records)
curve_df = pd.DataFrame(curve_records)

for label in sorted(curve_df["label"].unique()):
    fit = fit_log_chord_curve(curve_df[curve_df["label"] == label], NY)
    fit_records.append({"label": label, "decoh": label == "decoh_on", **fit})
fit_df = pd.DataFrame(fit_records)

table_paths = {
    "negativity": save_dataframe_pair(negativity_df, "final_state_strip_negativity_by_y0"),
    "curve": save_dataframe_pair(curve_df, "final_state_y0_averaged_negativity_curve"),
    "fit": save_dataframe_pair(fit_df, "final_state_log_chord_fit_summary"),
}
write_json(OUTPUT_DIR / "table_manifest.json", table_paths)
display(fit_df)
display(curve_df)

In [ ]:
# Plot controls: y0-averaged logarithmic negativity versus log chord length
REGENERATE_NEGATIVITY_FIGURE = True
SAVE_NEGATIVITY_FIGURE = True
NEGATIVITY_FIG_PATH = OUTPUT_DIR / "final_state_y0_averaged_negativity_log_chord.png"
FIG_WIDTH = 3.375
FIG_HEIGHT = 2.7
TITLE = f"N{NX}x{NY} Markov-channel final state"
X_LABEL = r"$\log[(N_y/\pi)\sin(\pi A_y/N_y)]$"
Y_LABEL = r"$\overline{E_N(A_y)}_{y_0}$"
LEGEND_LOC = "best"
X_SCALE = "linear"
Y_SCALE = "linear"
X_LIMITS = None
Y_LIMITS = None
COLORS = {"decoh_on": "tab:blue", "decoh_off": "tab:orange"}
MARKERS = {"decoh_on": "o", "decoh_off": "s"}

if REGENERATE_NEGATIVITY_FIGURE:
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT), constrained_layout=True)
    for label in ["decoh_on", "decoh_off"]:
        sub = curve_df[curve_df["label"] == label].sort_values("Ay")
        fit = fit_df[fit_df["label"] == label].iloc[0].to_dict()
        ax.errorbar(
            sub["log_chord"],
            sub["log_negativity_mean"],
            yerr=sub["log_negativity_sem"],
            marker=MARKERS[label],
            ms=3,
            lw=1.0,
            capsize=2,
            color=COLORS[label],
            label=f"{label}: m={fit['slope']:.4g} +/- {fit['slope_err']:.2g}, R^2={fit['r2']:.3f}",
        )
        x_line = np.linspace(float(sub["log_chord"].min()), float(sub["log_chord"].max()), 200)
        y_line = float(fit["slope"]) * x_line + float(fit["intercept"])
        ax.plot(x_line, y_line, ls="--", lw=0.9, color=COLORS[label])
    ax.set_title(TITLE, fontsize=8)
    ax.set_xlabel(X_LABEL, fontsize=8)
    ax.set_ylabel(Y_LABEL, fontsize=8)
    ax.set_xscale(X_SCALE)
    ax.set_yscale(Y_SCALE)
    if X_LIMITS is not None:
        ax.set_xlim(*X_LIMITS)
    if Y_LIMITS is not None:
        ax.set_ylim(*Y_LIMITS)
    ax.tick_params(labelsize=7)
    ax.legend(loc=LEGEND_LOC, fontsize=6, frameon=False)
    if SAVE_NEGATIVITY_FIGURE:
        fig.savefig(NEGATIVITY_FIG_PATH, dpi=300, bbox_inches="tight")
        print(f"saved {NEGATIVITY_FIG_PATH}")
    plt.show()

In [ ]:
# Plot controls: spatially resolved unit-cell charge heatmap videos
REGENERATE_CHARGE_VIDEOS = True
DISPLAY_CHARGE_VIDEOS = True
SECONDS_PER_CYCLE = 3
VIDEO_FPS = 1
VIDEO_DPI = 300
VIDEO_EXT = ".mp4"

video_records = []
if REGENERATE_CHARGE_VIDEOS:
    for label, payload in histories.items():
        model = payload["model"]
        title = f"{label} unit-cell charge, N{NX}x{NY}"
        video_path = OUTPUT_DIR / f"unit_cell_charge_heatmap_{label}{VIDEO_EXT}"
        record = make_charge_video(
            payload["G_hist"],
            nx=NX,
            ny=NY,
            title=title,
            output_path=video_path,
            seconds_per_cycle=SECONDS_PER_CYCLE,
            fps=VIDEO_FPS,
            dw_loc=getattr(model, "DW_loc", None),
        )
        record.update({"label": label, "decoh": label == "decoh_on"})
        video_records.append(record)
        print(record)
        if DISPLAY_CHARGE_VIDEOS:
            if record["path"].endswith(".mp4"):
                display(Video(record["path"], embed=True))
            elif record.get("html") is not None:
                display(record["html"])

video_df = pd.DataFrame(video_records)
if not video_df.empty:
    save_dataframe_pair(video_df, "charge_heatmap_video_manifest")
video_df

In [ ]:
# Summary
summary = {
    "config": RUN_CONFIG,
    "run_records": run_records,
    "fit_summary": fit_df.to_dict(orient="records"),
    "table_paths": table_paths,
    "output_dir": str(OUTPUT_DIR),
    "negativity_figure": str(NEGATIVITY_FIG_PATH),
    "video_records": video_records,
    "created": timestamp(),
}
summary_path = OUTPUT_DIR / "analysis_summary.json"
write_json(summary_path, summary)
print(f"summary saved to {summary_path}")
display(fit_df)